This script extracts potential location names from each row of the haunted places dataset using spaCy’s Named Entity Recognition and noun phrase parsing. It queries a local Lucene Geo Gazetteer to retrieve latitude and longitude for each detected location and adds the location, latitude, and longitude columns to the haunted places v2 dataset. 


Tools Used:
- spaCy for Named Entity Recognition (NER)
- Lucene Geo Gazetteer via REST API for converting locations to geographic coordinates
- Tika GeoTopicParser to enhance spatial metadata
- Pandas for data processing



In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [224]:
import pandas as pd
import spacy
import requests
import re

# Load  model
nlp = spacy.load("en_core_web_sm")

# Gazetteer REST API endpoint
GAZETTEER_API = "http://localhost:8765/api/search"

#  dataset
file_path = "/Users/rehamatai/dsci_550_a1/data/processed/haunted_places_features_added_v2.tab"
df = pd.read_csv(file_path, sep="\t")

# Add columns to DataFrame 
for col in ['Locations', 'Latitudes', 'Longitudes']:
    if col not in df.columns:
        df[col] = None

#  clean location names
def clean_location(loc):
    loc = loc.strip()
    loc = re.sub(r"^(the|nearby)\s+", "", loc, flags=re.IGNORECASE)
    loc = re.sub(r"\s*[-–—]\s*$", "", loc)
    loc = re.sub(r"\bUnited\b", "United States", loc)
    return loc.strip()

# Query Gazetteer
def get_lat_lon(location):
    try:
        res = requests.get(GAZETTEER_API, params={"s": location})
        if res.status_code == 200:
            data = res.json()
            if location in data and data[location]:
                loc_data = data[location][0]
                return loc_data["name"], loc_data["latitude"], loc_data["longitude"]
    except Exception as e:
        print(f" Error querying '{location}': {e}")
    return location, None, None

# Loop through all rows in the DataFrame
for idx, row in df.iterrows():
    combined_text = " ".join(str(row[col]) for col in ['City', 'Country', 'State', 'Description', 'Location'] if pd.notnull(row[col]))
    doc = nlp(combined_text)

    locations = set()

    for ent in doc.ents:
        if ent.label_ in {"GPE", "LOC", "FAC", "ORG"}:
            loc = clean_location(ent.text)
            if len(loc.split()) <= 4 and not any(char.isdigit() for char in loc):
                locations.add(loc)

    for chunk in doc.noun_chunks:
        if any(word.ent_type_ in {"GPE", "LOC", "FAC"} for word in chunk):
            loc = clean_location(chunk.text)
            if len(loc.split()) <= 4 and not any(char.isdigit() for char in loc):
                locations.add(loc)

    geo_results = []
    for loc in sorted(locations):
        name, lat, lon = get_lat_lon(loc)
        if lat is not None and lon is not None:
            geo_results.append((name, lat, lon))

    # Save results 
    if geo_results:
        df.at[idx, "Locations"] = ", ".join(res[0] for res in geo_results)
        df.at[idx, "Latitudes"] = ", ".join(str(res[1]) for res in geo_results)
        df.at[idx, "Longitudes"] = ", ".join(str(res[2]) for res in geo_results)

    if idx % 10 == 0:
        print(f"✅ Row {idx + 1}/{len(df)} processed")

# view results
print("\n Sample Results:")
print(df[['Locations', 'Latitudes', 'Longitudes']].head())

# Save results back to the file
df.to_csv(file_path, sep="\t", index=False)
print(f"\n Final results saved back to: {file_path}")


✅ Row 1/10992 processed
✅ Row 11/10992 processed
✅ Row 21/10992 processed
✅ Row 31/10992 processed
✅ Row 41/10992 processed
✅ Row 51/10992 processed
✅ Row 61/10992 processed
✅ Row 71/10992 processed
✅ Row 81/10992 processed
✅ Row 91/10992 processed
✅ Row 101/10992 processed
✅ Row 111/10992 processed
✅ Row 121/10992 processed
✅ Row 131/10992 processed
✅ Row 141/10992 processed
✅ Row 151/10992 processed
✅ Row 161/10992 processed
✅ Row 171/10992 processed
✅ Row 181/10992 processed
✅ Row 191/10992 processed
✅ Row 201/10992 processed
✅ Row 211/10992 processed
✅ Row 221/10992 processed
✅ Row 231/10992 processed
✅ Row 241/10992 processed
✅ Row 251/10992 processed
✅ Row 261/10992 processed
✅ Row 271/10992 processed
✅ Row 281/10992 processed
✅ Row 291/10992 processed
✅ Row 301/10992 processed
✅ Row 311/10992 processed
✅ Row 321/10992 processed
✅ Row 331/10992 processed
✅ Row 341/10992 processed
✅ Row 351/10992 processed
✅ Row 361/10992 processed
✅ Row 371/10992 processed
✅ Row 381/10992 process

In [212]:
#Script aims to counts how often each location name appears in the "Locations" column of the dataset

import pandas as pd
from collections import Counter


df_path = "/Users/rehamatai/dsci_550_a1/data/processed/haunted_places_features_added_v2.tab"
df = pd.read_csv(df_path, sep="\t")


all_locations = df["Locations"].dropna().apply(lambda x: [loc.strip() for loc in x.split(",")])


flat_list = [loc for sublist in all_locations for loc in sublist]
location_counts = Counter(flat_list)


location_freq_df = pd.DataFrame(location_counts.items(), columns=["Location", "Frequency"])
location_freq_df.sort_values(by="Frequency", ascending=False, inplace=True)


print(location_freq_df.head(20))


            Location  Frequency
648       California        739
321            Texas        538
161             Ohio        410
3           Michigan        394
331     Pennsylvania        361
1758        New York        333
1836        Missouri        286
4604         Florida        278
483          Indiana        245
2404        Kentucky        237
790    Massachusetts        219
1928         Georgia        202
4501       Tennessee        196
1683   West Virginia        187
365       Washington        185
1656  North Carolina        170
1757      New Jersey        170
2786     Connecticut        158
1742        Oklahoma        154
1667        Maryland        150


In [214]:
# Script aims to find what kinds of entities are most associated with top cities


from collections import defaultdict
import pandas as pd

df = pd.read_csv("/Users/rehamatai/dsci_550_a1/data/processed/haunted_places_features_added_v2.tab", sep="\t")


city_entity_map = defaultdict(list)

for idx, row in df.iterrows():
    city = row.get("City")
    locations = row.get("Locations")
    if pd.notnull(city) and pd.notnull(locations):
        for loc in locations.split(","):
            loc = loc.strip()
            if loc and loc.lower() != city.lower():
                city_entity_map[city].append(loc)

for city in list(city_entity_map.keys())[:5]:  # limit output to first 5 cities
    print(f"\n📍 City: {city}")
    entities = pd.Series(city_entity_map[city])
    print(entities.value_counts().head(5))



📍 City: Ada
Oklahoma                     3
Old Ada Cemetery             1
Egypt Valley Country Club    1
Town of Honey Creek          1
Michigan                     1
Name: count, dtype: int64

📍 City: Addison
Michigan    1
Name: count, dtype: int64

📍 City: Adrian
Michigan                    2
Town of Sand Creek          1
Siena Heights University    1
Mei King Mansion            1
Young County                1
Name: count, dtype: int64

📍 City: Albion
Michigan                            2
Albion College Historical Marker    1
Name: count, dtype: int64

📍 City: Algoma Township
Algoma              1
Michigan            1
City of Rockford    1
Rogue River         1
Name: count, dtype: int64
